# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a reproducible workflow for loading and exploring the FAIR\u00b2 protein mass spectrometry dataset using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant pandas --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # This is an object, not a dict
# Display key summary metadata
print(f"Title: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')}")
print(f"Version: {getattr(metadata, 'version', 'N/A')}")
print(f"License: {getattr(metadata, 'license', 'N/A')}")


## 2. Data Overview
Review available record sets and fields (all referenced by their `@id`).

**Note:** The Croissant schema for this dataset programmatically exposes record sets and their fields. We'll enumerate all record sets, their IDs, and the fields in each.

In [ ]:
# List all RecordSets and their fields
record_sets = list(dataset.record_sets)
print(f"Available record sets (by @id):\n")

record_set_ids = []
for rs in record_sets:
    print(f"- RecordSet: {rs.id}")
    record_set_ids.append(rs.id)
    field_ids = [field.id for field in rs.fields]
    print(f"  Fields (@id): {field_ids}")
    print()
# For demonstration, let's pick the first one for further examples
if len(record_set_ids) > 0:
    main_record_set_id = record_set_ids[0]
else:
    raise ValueError("No record sets found in this dataset.")

## 3. Data Extraction
Load records from a specific record set into a DataFrame for analysis. All operations will use `@id` notation for record sets and fields.

In [ ]:
# Extract data from all available record sets
dataframes = {}

for rec_set_id in record_set_ids:
    print(f"Loading RecordSet: {rec_set_id}")
    records_gen = dataset.records(record_set=rec_set_id)  # Generator of records (dicts)
    try:
        df = pd.DataFrame(records_gen)
        dataframes[rec_set_id] = df
        print(f"  Loaded shape: {df.shape}")
    except Exception as e:
        print(f"  Failed to load into DataFrame: {e}")

# Display the fields of the main RecordSet (first one listed)
print(f"\nColumns for RecordSet {main_record_set_id}:")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps using field `@id`s.

- Filter for records with high coverage or abundance
- Normalize the data
- Group by sample or annotation fields, if available

*(Adjust field IDs as needed based on actual dataset schema discovered above)*

In [ ]:
# Please update these IDs based on the output above as needed:
# Example assumptions for field @id entries (as discovered in the overview above)

# Try to select a numeric field for analysis
df = dataframes[main_record_set_id]

# Heuristically pick the first numeric column
numeric_cols = df.select_dtypes(include=['number', 'float', 'int']).columns.tolist()
print(f"Numeric fields detected: {numeric_cols}")

if len(numeric_cols) == 0:
    print("No numeric fields available for analysis.")
else:
    # Pick the most relevant, e.g., coverage or abundance field
    # e.g. '@id': 'coverage', in actual dataset it could be '@id': 'cr:coverage' or similar
    numeric_field_id = numeric_cols[0]
    print(f"Proceeding with numeric field: {numeric_field_id}")

    threshold = df[numeric_field_id].mean()  # Use the mean as threshold for demonstration
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (mean):")
    print(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try grouping by a categorical variable (e.g. 'description' or a sample/annotation field)
    possible_group_fields = df.select_dtypes(include='object').columns.tolist()
    group_field = possible_group_fields[0] if len(possible_group_fields) > 0 else None

    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().to_frame()
        print(f"Grouped data by {group_field} (top groups shown):")
        print(grouped_df.head())

## 5. Visualization
Visualize the distribution of the primary numeric field and show any group-based statistics.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize the distribution of the normalized numeric field
if len(numeric_cols) > 0:
    plt.figure(figsize=(7,4))
    sns.histplot(filtered_df[f"{numeric_field_id}_normalized"], bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id} (normalized)")
    plt.xlabel(f"{numeric_field_id} (normalized)")
    plt.ylabel("Count")
    plt.show()

    if group_field:
        plt.figure(figsize=(10,4))
        # Show means of the normalized values for each group (top 10)
        group_means = filtered_df.groupby(group_field)[f"{numeric_field_id}_normalized"].mean().sort_values().head(10)
        group_means.plot(kind='barh')
        plt.title(f"Top {group_field} groups (mean normalized {numeric_field_id})")
        plt.xlabel(f"Mean normalized {numeric_field_id}")
        plt.tight_layout()
        plt.show()

## 6. Conclusion
This notebook demonstrated how to use the [mlcroissant](https://mlcroissant.readthedocs.io/) library to load, examine, filter, and visualize a complex biomedical dataset defined by a Croissant schema. 

**Key learnings:**
- The dataset provides protein-level records with rich metadata.
- We identified all record sets and performed example numeric filtering and normalization.
- Standard EDA steps and visualizations can be performed directly on loaded dataframes.

> Explore other fields and compound queries with advanced `mlcroissant` syntax for greater insight.
